# PII and compliance discovery in plain SQL, with typed answers from Jev

`ai_parse` in Databricks and `prompt()` in MotherDuck put an LLM inside a query. Both hand back **free text**, which you then have to parse, validate and hope is stable.

Here the same idea returns a **typed struct**. One call per *column* (not per row) asks several questions at once, and every answer lands as its own column in a catalog table:

| question | Jev type | becomes |
|---|---|---|
| is it personal data? | `noul` (0..1) | `sensitive` |
| which kind? | `choice` + a probability per option | `category`, `category_conf`, `runner_up` |
| does GDPR / CCPA-CPRA / HIPAA / LGPD / PIPEDA / DPDP apply? | one `noul` each | `regime_*` |

What to *do* about a column (keep / mask / hash / drop / review) is **deterministic policy**, not a model answer. The AI picks the category; the rules decide the action.

Data: the synthetic **Claimwise** healthcare seeds (7 tables, 50 columns, no real people). Control: job-scout's public hiring data, which should come back clean. Needs `TYPESAFE_API_KEY`; without it a keyword mock runs so you can see the plumbing (its answers are *not* meaningful).

In [1]:
import duckdb, pandas as pd
from jev_pii import make_client, register, pii_questions
from jev_pii.catalog import sample_columns, build_catalog, deep_dive, build_silver
from jev_pii.client import load_dotenv
from jev_pii.score import score
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_colwidth", 60)

load_dotenv(".env")                       # TYPESAFE_API_KEY=... (git-ignored)
jev = make_client()
print("client:", jev.name, jev.model)

client: jev jev-1.14.0


## 1. Bronze: the raw seeds, untouched

In [2]:
TABLES = ["users", "encounters", "claims", "activities", "collections", "facilities", "health_plans"]
con = duckdb.connect()
for t in TABLES:
    con.execute(f"CREATE TABLE {t} AS SELECT * FROM read_csv_auto('data/raw/{t}.csv')")
con.sql("SELECT table_name, estimated_size AS rows, column_count FROM duckdb_tables() ORDER BY 1")

table_nameVARCHAR,rowsBIGINT,column_countBIGINT
activities,17529,8
claims,5000,8
collections,3054,7
encounters,8000,9
facilities,12,5
health_plans,24,4
users,1600,9


## 2. The function

`jev_pii(table, column, samples)` is a DuckDB Python UDF that returns a struct. The same call you would make by hand:

In [3]:
register(con, jev)          # register(con, jev, redact=True) sends value *shapes* instead of values
con.sql("""
SELECT r.sensitive, r.category, r.category_conf, r.runner_up, r.regime_hipaa, r.regime_gdpr
FROM (SELECT jev_pii('users', 'contactInfo', ['321.581.9600', '001-486-537-9402x654', '(555) 555-0101']) AS r)
""")

sensitiveDOUBLE,categoryVARCHAR,category_confDOUBLE,runner_upVARCHAR,regime_hipaaDOUBLE,regime_gdprDOUBLE
0.79,PHONE_NUMBER,1.0,CREDIT_CARD_NUMBER,0.92,0.91


## 3. Catalog: sample each column, ask once, keep every answer

`sample_columns` takes a reproducible random sample of 20 distinct values per column. `build_catalog` is one SQL statement that calls `jev_pii` per column. If Jev fails or is unreachable the answer is `NULL` and the column is parked for review. It never blocks the pipeline.

In [4]:
sample_columns(con, TABLES)
build_catalog(con)
con.sql("SELECT table_name, column_name, category, round(sensitive,2) AS sensitive, round(category_conf,2) AS conf, runner_up, regimes, action "
        "FROM column_catalog WHERE action <> 'keep' ORDER BY table_name, ordinal").df()

,table_name,column_name,category,sensitive,conf,runner_up,regimes,action
0,activities,diagnosisCode,HEALTH_CONDITION,0.74,0.83,NONE,"[GDPR, CCPA_CPRA, HIPAA, LGPD, PIPEDA, DPDP_INDIA]",mask
1,claims,patientID,HEALTH_RECORD_ID,0.94,0.99,NONE,"[GDPR, CCPA_CPRA, HIPAA, LGPD, PIPEDA, DPDP_INDIA]",hash
2,encounters,encounterID,NONE,0.59,0.65,HEALTH_RECORD_ID,[],review
3,encounters,patientID,HEALTH_RECORD_ID,0.93,0.99,NONE,"[GDPR, CCPA_CPRA, HIPAA, LGPD, PIPEDA, DPDP_INDIA]",hash
4,encounters,doctorID,NONE,0.71,0.64,EMPLOYMENT_INFO,[],review
5,facilities,location,PHYSICAL_ADDRESS,0.72,0.92,NONE,"[GDPR, CCPA_CPRA, HIPAA, LGPD, PIPEDA]",mask
6,users,id,NONE,0.60,0.92,OTHER_SENSITIVE,[],review
7,users,userID,LOGIN_CREDENTIAL,0.82,0.51,OTHER_SENSITIVE,"[GDPR, CCPA_CPRA, HIPAA, LGPD, PIPEDA, DPDP_INDIA]",drop
8,users,name,PERSON_NAME,0.99,1.00,CREDIT_CARD_NUMBER,"[GDPR, CCPA_CPRA, HIPAA, LGPD, PIPEDA, DPDP_INDIA]",mask
9,users,dateOfBirth,DATE_OF_BIRTH,0.99,1.00,CREDIT_CARD_NUMBER,"[GDPR, CCPA_CPRA, HIPAA, LGPD, PIPEDA, DPDP_INDIA]",mask


## 4. Deep dive on the weak spots

Columns with low category confidence, or a `sensitive` score in the 0.3-0.7 dead zone, are re-asked with a bigger sample (60 values).

In [5]:
deep_dive(con)

[('users', 'id'),
 ('users', 'userID'),
 ('users', 'userType'),
 ('users', 'specialty'),
 ('users', 'role'),
 ('encounters', 'encounterID'),
 ('encounters', 'doctorID'),
 ('claims', 'payerID'),
 ('activities', 'procedureCode'),
 ('facilities', 'contactInfo')]

## 5. Context matters: same column name, different verdict

`users.contactInfo` is a person's phone. `facilities.contactInfo` is a business line. Name-based rules and regexes cannot tell them apart. Look at what Jev made of each:

In [6]:
con.sql("SELECT table_name, column_name, category, round(sensitive,2) AS sensitive, action "
        "FROM column_catalog WHERE column_name = 'contactInfo'").df()

,table_name,column_name,category,sensitive,action
0,users,contactInfo,PHONE_NUMBER,0.88,mask
1,facilities,contactInfo,PHONE_NUMBER,0.33,keep


## 6. Score it

Ground truth is hand-labelled in `data/ground_truth.csv` (`facilities` is a deliberate trap: names, addresses and phones that belong to a *business*). Columns I could not call with confidence are marked `ambiguous` and excluded from the score, not counted for or against.

Regime agreement is checked against `config/compliance_matrix.yaml`, the mapping table from `agent-pii-discovery`. **The matrix is the answer key, not the runtime source**: Jev replaces maintaining it per column, and the score shows where the two disagree.

In [7]:
result = score(con)
{k: v for k, v in result.items() if k not in ("false_positives", "misses")}

{'columns_scored': 40,
 'ambiguous_excluded': 10,
 'precision': 0.857,
 'recall': 0.857,
 'f1': 0.857,
 'category_accuracy': 0.857,
 'regime_agreement_with_matrix': 0.694}

In [8]:
print('false positives:', result['false_positives']); print('misses:', result['misses'])

false positives: [['facilities', 'location', 'PHYSICAL_ADDRESS']]
misses: [['activities', 'procedureCode', 'HEALTH_CONDITION']]


Read the misses and false positives as a to-do list, not a verdict: seven labelled PII columns is a small sample. The review queue (`action = 'review'`) is where the system says "I am not sure", which is the point.

## 7. Silver: policy applied, one view per table

In [9]:
build_silver(con, TABLES)
print(con.sql("SELECT name, dateOfBirth, contactInfo FROM users LIMIT 3").df().to_string())
print(con.sql("SELECT * FROM silver_users LIMIT 3").df().to_string())

               name dateOfBirth           contactInfo
0      Allison Hill  1965-02-10          321.581.9600
1  Kristina Baldwin  1994-10-27  001-486-537-9402x654
2     Brandon Davis  1984-06-17     (794)507-8161x849
     id userType name dateOfBirth contactInfo specialty department role
0  None     None  ***         ***         ***       ***       None  ***
1  None     None  ***         ***         ***       ***       None  ***
2  None     None  ***         ***         ***       ***       None  ***


## 8. Control: job-scout (public job postings, no personal data expected)

In [10]:
con.execute("CREATE TABLE ats AS SELECT * FROM read_parquet('data/raw/ats_raw_trends.parquet')")
sample_columns(con, ["ats"])
build_catalog(con)
con.sql("SELECT column_name, category, round(sensitive,2) AS sensitive, action FROM column_catalog ORDER BY ordinal").df()

,column_name,category,sensitive,action
0,company_name,NONE,0.03,keep
1,ats_platform,NONE,0.03,keep
2,req_id,NONE,0.19,keep
3,title,EMPLOYMENT_INFO,0.24,keep
4,department,EMPLOYMENT_INFO,0.07,keep
5,team,EMPLOYMENT_INFO,0.07,keep
6,employment_type,EMPLOYMENT_INFO,0.60,mask
7,location,PHYSICAL_ADDRESS,0.46,keep
8,work_mode,EMPLOYMENT_INFO,0.35,keep
9,comp_summary,NONE,0.04,keep


## Cost

In [11]:
print(f"{jev.calls} API calls, {jev.cache_hits} cache hits, {jev.input_tokens} input / {jev.output_tokens} output tokens")
con.close()

19 API calls, 60 cache hits, 23197 input / 6843 output tokens
